# Pneumonia Detection - YOLO11m (Detection)

Runs the **yolo11** model (Ultralytics YOLO11m) through **phases 3->8** (baseline -> hyperparameter optimization -> retrain -> explainability -> evaluation -> demo) and writes `artifacts/yolo11_report.json`.

### Before you run
1. **Add the prepped data**: *Add Input* -> the dataset created by `00_prepare_dataset` (it contains `yolo_dataset/`). The setup cell auto-detects it; if your slug differs, set `PREPPED_INPUT` in the first code cell (the printed `Attached inputs` list shows the name).
2. **Enable GPU**: *Settings -> Accelerator -> GPU* (P100 or T4) and keep **Internet ON** (YOLO11m auto-downloads `yolo11m.pt`).
3. YOLO11m is a medium model, so the baseline batch size is 8 (safe on a 16GB T4). On a P100 you can raise it.
4. If the GitHub repo is private: make it public, use a token in `REPO_URL`, or upload the `ai/` folder as a Kaggle dataset and `sys.path` to it instead of cloning.

When done, download `yolo11_report.json` and the checkpoint from the **Output** tab and send them back.

In [ ]:
import os, sys, shutil, glob

# ===== EDIT IF NEEDED =====
REPO_URL = "https://github.com/202201638/Graduation_project_Fully_team_2026.git"
# Folder you attached from the 00_prepare_dataset output (the prepped yolo_dataset).
PREPPED_INPUT = "/kaggle/input/rsna-prepped-yolo-dataset"
# ==========================

WORK = "/kaggle/working"
os.environ["YOLO_DATASET_DIR"] = f"{WORK}/yolo_dataset"
os.environ["ARTIFACT_DIR"]     = f"{WORK}/artifacts"
os.environ["RUNS_DIR"]         = f"{WORK}/runs"
os.environ["PNG_DIR"]          = f"{WORK}/png_images"

# Always re-clone. /kaggle/working persists across kernel restarts, so a stale checkout
# from an earlier run would otherwise shadow the latest main and you'd keep running old code.
REPO_DIR = f"{WORK}/repo"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
os.system(f"GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 {REPO_URL} {REPO_DIR}")
AI_DIR = f"{REPO_DIR}/ai"
sys.path.insert(0, AI_DIR)
os.chdir(AI_DIR)

# deps that may be missing / outdated on Kaggle (torch/torchvision already present).
# A recent ultralytics is required for YOLO11 (yolo11m.pt).
os.system("pip install -q -U ultralytics torchmetrics pydicom")

# Locate the prepped dataset regardless of how Kaggle nested it, then copy it into the
# writable working dir (Ultralytics writes .cache files next to labels, which fails on
# the read-only /kaggle/input).
print("Attached inputs:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "none")

def _find_yolo_root(base):
    cands = [base, os.path.join(base, "yolo_dataset")]
    cands += glob.glob(os.path.join(base, "*", "yolo_dataset"))
    cands += glob.glob(os.path.join(base, "*"))
    for c in cands:
        if os.path.isdir(os.path.join(c, "train", "images")):
            return c
    return None

SRC = _find_yolo_root(PREPPED_INPUT)
assert SRC, (f"Could not find train/images under {PREPPED_INPUT}. "
             f"Contents: {os.listdir(PREPPED_INPUT) if os.path.isdir(PREPPED_INPUT) else 'PATH MISSING'}. "
             f"Attach the prepped dataset and set PREPPED_INPUT to its path.")

DST = os.environ["YOLO_DATASET_DIR"]
if not os.path.isdir(os.path.join(DST, "train", "images")):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
print("Dataset ready:", SRC, "->", DST, "| splits:", os.listdir(DST))

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU)")

from src.model_pipeline import run_model_pipeline

MODEL = "yolo11"

# Knobs - reduce if a Kaggle session is running long:
EPOCHS       = None   # None = sensible default for this model (40, early-stopped)
POPULATION   = 3      # phase-4 (PSO/GWO/SA) population
ITERATIONS   = 2      # phase-4 iterations
PROXY_EPOCHS = 1      # phase-4 proxy training epochs per candidate

report = run_model_pipeline(
    MODEL,
    epochs=EPOCHS,
    run_baseline=True,
    optimize=True,
    population=POPULATION,
    iterations=ITERATIONS,
    proxy_epochs=PROXY_EPOCHS,
)

## Results, parameters and metrics

In [ ]:
import json, os
from IPython.display import Image, display

report_path = f"/kaggle/working/artifacts/{MODEL}_report.json"
with open(report_path) as f:
    report = json.load(f)

# print everything except the verbose phase-4 algorithm trace
slim = {k: v for k, v in report.items() if k != "phase4_optimization"}
print(json.dumps(slim, indent=2, default=str))

for key in ("phase6_explainability", "phase8_demo"):
    info = report.get(key, {}) or {}
    img = info.get("image") or info.get("output_image")
    if img and os.path.exists(img):
        print("\n", key, "->", img)
        display(Image(img))

## Download artifacts for hand-back

In [ ]:
import os, glob
print("Hand these back (Output tab):")
print(" - report  :", f"/kaggle/working/artifacts/{MODEL}_report.json")
for p in sorted(glob.glob("/kaggle/working/artifacts/checkpoints/*.pt")):
    print(" - weights :", p, f"({os.path.getsize(p)/1e6:.1f} MB)")
for p in sorted(glob.glob("/kaggle/working/runs/**/weights/best.pt", recursive=True)):
    print(" - yolo    :", p, f"({os.path.getsize(p)/1e6:.1f} MB)")